# Intermediate 08 — Workload Assurance & Runtime Attestation for Agents

## Scenario

The enterprise registry says `claims-agent` is approved. We must prove that a request actually comes from the approved production workload and artifact.

```text
agent registry
     ↓
approved logical agent
     ↓
approved SPIFFE identity
     ↓
attested workload
     ↓
approved artifact + provenance
     ↓
runtime posture
     ↓
authorization
```

The notebook first builds the concepts locally, then gives exercises for a real SPIRE deployment.


In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse
import hashlib, json, uuid, copy
import jwt

def now():
    return datetime.now(timezone.utc)


## 1 — Parse and validate SPIFFE IDs

In [ ]:
def parse_spiffe_id(value):
    u=urlparse(value)
    if u.scheme!="spiffe":
        raise ValueError("SPIFFE ID must use spiffe scheme")
    if not u.netloc:
        raise ValueError("SPIFFE trust domain required")
    if u.query or u.fragment:
        raise ValueError("SPIFFE ID cannot contain query/fragment")
    return {"trust_domain":u.netloc,"path":u.path or "/"}

sid="spiffe://corp.example/prod/agents/claims-agent"
print(parse_spiffe_id(sid))


## 2 — Trust domains

In [ ]:
TRUST_DOMAINS={
 "corp.example":{
   "environment":"enterprise",
   "federates_with":{"partner.example"}
 },
 "partner.example":{
   "environment":"partner",
   "federates_with":{"corp.example"}
 }
}
print(json.dumps(TRUST_DOMAINS,indent=2))


## 3 — Registration entries and selectors

In [ ]:
REGISTRATION=[
 {
   "spiffe_id":"spiffe://corp.example/prod/agents/claims-agent",
   "selectors":{
     "k8s:namespace:claims",
     "k8s:service-account:claims-agent",
     "cluster:prod"
   }
 }
]

def select_identity(observed):
    matches=[]
    for entry in REGISTRATION:
        if entry["selectors"] <= set(observed):
            matches.append(entry["spiffe_id"])
    return matches

observed={
 "k8s:namespace:claims",
 "k8s:service-account:claims-agent",
 "cluster:prod"
}
print(select_identity(observed))


## 4 — Selector weakness

In [ ]:
weak_entry={
 "spiffe_id":"spiffe://corp.example/prod/agents/claims-agent",
 "selectors":{"k8s:namespace:claims"}
}

attacker={
 "k8s:namespace:claims",
 "k8s:service-account:unrelated"
}

print("weak match:",weak_entry["selectors"] <= attacker)
print("strong match:",REGISTRATION[0]["selectors"] <= attacker)


## 5 — Separate node and workload attestation

In [ ]:
attestation={
 "node":{
   "node_id":"node:prod-17",
   "attested":True,
   "evidence":"cloud-instance-identity",
   "observed_at":now()
 },
 "workload":{
   "selectors":sorted(observed),
   "attested":True,
   "observed_at":now()
 }
}
print(json.dumps(attestation,indent=2,default=str))


## 6 — Simulated X.509-SVID lifecycle

In [ ]:
@dataclass
class X509SVID:
    spiffe_id: str
    serial: str
    issued_at: datetime
    expires_at: datetime

def issue_x509_svid(spiffe_id,ttl_minutes=30):
    t=now()
    return X509SVID(
      spiffe_id=spiffe_id,
      serial=str(uuid.uuid4()),
      issued_at=t,
      expires_at=t+timedelta(minutes=ttl_minutes)
    )

x509_svid=issue_x509_svid(sid)
print(asdict(x509_svid))


This object teaches lifecycle semantics; it is **not** a replacement for a real SPIFFE X.509-SVID. The live exercise later uses SPIRE's Workload API.

## 7 — Rotation

In [ ]:
old=x509_svid
new=issue_x509_svid(sid)
print("same identity:",old.spiffe_id==new.spiffe_id)
print("new credential:",old.serial!=new.serial)


## 8 — Do not pin leaf credentials

In [ ]:
# Bad policy:
BAD_ALLOWED_SERIALS={old.serial}

# Better policy reasons over the stable SPIFFE identity and trusted issuer/bundle.
APPROVED_SPIFFE_IDS={sid}

print(new.serial in BAD_ALLOWED_SERIALS)
print(new.spiffe_id in APPROVED_SPIFFE_IDS)


## 9 — Simulated JWT-SVID

In [ ]:
JWT_KEY="training-only-key"
JWT_ISSUER="https://spire.example"

def issue_jwt_svid(spiffe_id,audience,ttl_seconds=120):
    t=now()
    return jwt.encode({
      "iss":JWT_ISSUER,
      "sub":spiffe_id,
      "aud":audience,
      "iat":int(t.timestamp()),
      "exp":int((t+timedelta(seconds=ttl_seconds)).timestamp()),
      "jti":str(uuid.uuid4())
    },JWT_KEY,algorithm="HS256")

token=issue_jwt_svid(sid,"payments-api")
print(jwt.decode(token,JWT_KEY,algorithms=["HS256"],
                 audience="payments-api",issuer=JWT_ISSUER))


## 10 — JWT audience confusion

In [ ]:
try:
    jwt.decode(token,JWT_KEY,algorithms=["HS256"],
               audience="claims-api",issuer=JWT_ISSUER)
except Exception as e:
    print("REJECTED:",type(e).__name__)


## 11 — Bearer replay risk

In [ ]:
print("A copied JWT-SVID remains a bearer credential until it expires.")
print("Mitigate with short TTL, strict audience, transport security and minimal exposure.")


## 12 — Logical agent to workload binding

In [ ]:
AGENT_REGISTRY={
 "claims-agent":{
   "status":"approved",
   "allowed_spiffe_ids":{
     "spiffe://corp.example/prod/agents/claims-agent"
   }
 }
}

def bind_agent(logical_agent,spiffe_id):
    record=AGENT_REGISTRY.get(logical_agent)
    return bool(
      record
      and record["status"]=="approved"
      and spiffe_id in record["allowed_spiffe_ids"]
    )

print(bind_agent("claims-agent",sid))
print(bind_agent("claims-agent","spiffe://corp.example/dev/agents/claims-agent"))


## 13 — Artifact digests

In [ ]:
agent_artifact=b'''
name=claims-agent
version=4.3.1
policy=19
'''
digest="sha256:"+hashlib.sha256(agent_artifact).hexdigest()
print(digest)


## 14 — Approved artifact identity

In [ ]:
APPROVED_IMAGES={
 "claims-agent":{digest}
}

def image_approved(agent,image_digest):
    return image_digest in APPROVED_IMAGES.get(agent,set())

print(image_approved("claims-agent",digest))
print(image_approved("claims-agent","sha256:attacker"))


## 15 — Simplified SLSA provenance object

In [ ]:
provenance={
 "_type":"https://in-toto.io/Statement/v1",
 "subject":[{"name":"claims-agent","digest":{"sha256":digest.split(":")[1]}}],
 "predicateType":"https://slsa.dev/provenance/v1",
 "predicate":{
   "buildDefinition":{
     "buildType":"https://example.org/build/container/v1",
     "externalParameters":{
       "repository":"https://github.com/example/claims-agent",
       "ref":"refs/tags/v4.3.1"
     }
   },
   "runDetails":{
     "builder":{"id":"https://ci.example/builders/prod"}
   }
 }
}
print(json.dumps(provenance,indent=2))


## 16 — Provenance expectation checks

In [ ]:
EXPECTED={
 "builder":"https://ci.example/builders/prod",
 "repository":"https://github.com/example/claims-agent",
 "build_type":"https://example.org/build/container/v1"
}

def verify_provenance_expectations(p,artifact_digest):
    subject=p["subject"][0]["digest"]["sha256"]
    if subject != artifact_digest.split(":")[1]:
        return False,"artifact digest mismatch"
    pred=p["predicate"]
    if pred["runDetails"]["builder"]["id"] != EXPECTED["builder"]:
        return False,"unapproved builder"
    bd=pred["buildDefinition"]
    if bd["buildType"] != EXPECTED["build_type"]:
        return False,"unexpected build type"
    if bd["externalParameters"]["repository"] != EXPECTED["repository"]:
        return False,"unexpected source repository"
    return True,"expectations satisfied"

print(verify_provenance_expectations(provenance,digest))


Production verification must also authenticate the attestation envelope/signature. This lab deliberately separates **content expectations** from **cryptographic verification** so students do not confuse the two.

## 17 — Sigstore/Cosign verification policy model

In [ ]:
cosign_result={
 "signature_verified":True,
 "certificate_identity":"https://github.com/example/claims-agent/.github/workflows/release.yml@refs/tags/v4.3.1",
 "certificate_oidc_issuer":"https://token.actions.githubusercontent.com",
 "transparency_verified":True
}

SIGSTORE_POLICY={
 "issuer":"https://token.actions.githubusercontent.com",
 "identity_prefix":"https://github.com/example/claims-agent/"
}

def accept_cosign(r):
    return (
      r["signature_verified"]
      and r["transparency_verified"]
      and r["certificate_oidc_issuer"]==SIGSTORE_POLICY["issuer"]
      and r["certificate_identity"].startswith(SIGSTORE_POLICY["identity_prefix"])
    )

print(accept_cosign(cosign_result))


## 18 — Runtime posture

In [ ]:
posture={
 "spiffe_id":sid,
 "node_attested":True,
 "workload_attested":True,
 "image_digest":digest,
 "image_verified":True,
 "provenance_verified":True,
 "namespace":"claims",
 "service_account":"claims-agent",
 "agent_version":"4.3.1",
 "policy_version":"19",
 "observed_at":now(),
 "status":"active"
}
print(json.dumps(posture,indent=2,default=str))


## 19 — Drift detection

In [ ]:
EXPECTED_RUNTIME={
 "namespace":"claims",
 "service_account":"claims-agent",
 "agent_version":"4.3.1",
 "policy_version":"19"
}

def detect_drift(p):
    drift={}
    for k,v in EXPECTED_RUNTIME.items():
        if p.get(k)!=v:
            drift[k]={"expected":v,"observed":p.get(k)}
    if not image_approved("claims-agent",p["image_digest"]):
        drift["image_digest"]={"expected":"approved digest","observed":p["image_digest"]}
    return drift

print(detect_drift(posture))

tampered=copy.deepcopy(posture)
tampered["image_digest"]="sha256:evil"
print(detect_drift(tampered))


## 20 — Attestation freshness

In [ ]:
def posture_fresh(p,max_age_seconds=300):
    age=(now()-p["observed_at"]).total_seconds()
    return age <= max_age_seconds

print(posture_fresh(posture))

stale=copy.deepcopy(posture)
stale["observed_at"]=now()-timedelta(hours=2)
print(posture_fresh(stale))


## 21 — Workload-aware authorization

In [ ]:
TASK={
 "id":"task:483",
 "active":True,
 "resource":"claim:483",
 "actions":{"claim.read","claim.update"}
}

def authorize(agent,action,resource,p,task):
    reasons=[]
    if not bind_agent(agent,p["spiffe_id"]):
        reasons.append("agent/workload binding failed")
    if not p["node_attested"]:
        reasons.append("node not attested")
    if not p["workload_attested"]:
        reasons.append("workload not attested")
    if not p["image_verified"]:
        reasons.append("image not verified")
    if not p["provenance_verified"]:
        reasons.append("provenance not verified")
    if detect_drift(p):
        reasons.append("runtime drift")
    if not posture_fresh(p):
        reasons.append("posture stale")
    if p["status"]!="active":
        reasons.append("workload quarantined")
    if not task["active"]:
        reasons.append("task inactive")
    if action not in task["actions"]:
        reasons.append("action outside task")
    if resource != task["resource"]:
        reasons.append("resource outside task")
    return (not reasons),reasons

print(authorize("claims-agent","claim.update","claim:483",posture,TASK))


## 22 — Valid identity does not override authorization

In [ ]:
print(authorize(
 "claims-agent",
 "payment.create",   # not in task
 "claim:483",
 posture,
 TASK
))


## 23 — Runtime compromise/quarantine

In [ ]:
quarantined=copy.deepcopy(posture)
quarantined["status"]="quarantined"

print("SPIFFE identity still:",quarantined["spiffe_id"])
print(authorize("claims-agent","claim.read","claim:483",quarantined,TASK))


## 24 — Federation

In [ ]:
def can_authenticate_foreign(local_td,foreign_spiffe_id):
    foreign_td=parse_spiffe_id(foreign_spiffe_id)["trust_domain"]
    return foreign_td in TRUST_DOMAINS[local_td]["federates_with"]

partner="spiffe://partner.example/prod/tools/risk-engine"
print(can_authenticate_foreign("corp.example",partner))


## 25 — Federation is not authorization

In [ ]:
FOREIGN_AUTHZ={
 "spiffe://partner.example/prod/tools/risk-engine":{"risk.score"}
}

def authorize_foreign(spiffe_id,action):
    if not can_authenticate_foreign("corp.example",spiffe_id):
        return False
    return action in FOREIGN_AUTHZ.get(spiffe_id,set())

print(authorize_foreign(partner,"risk.score"))
print(authorize_foreign(partner,"payment.create"))


## 26 — Evidence record

In [ ]:
def evidence(agent,action,resource,p,task):
    allowed,reasons=authorize(agent,action,resource,p,task)
    return {
      "decision_id":str(uuid.uuid4()),
      "logical_agent":agent,
      "spiffe_id":p["spiffe_id"],
      "image_digest":p["image_digest"],
      "node_attested":p["node_attested"],
      "workload_attested":p["workload_attested"],
      "image_verified":p["image_verified"],
      "provenance_verified":p["provenance_verified"],
      "posture_fresh":posture_fresh(p),
      "action":action,
      "resource":resource,
      "decision":"allow" if allowed else "deny",
      "reasons":reasons
    }

print(json.dumps(evidence(
 "claims-agent","claim.update","claim:483",posture,TASK
),indent=2))


## 27 — Adversarial regression tests

In [ ]:
tests=[]

ok,_=authorize("claims-agent","claim.update","claim:483",posture,TASK)
tests.append(("approved runtime",ok,True))

bad=copy.deepcopy(posture)
bad["spiffe_id"]="spiffe://corp.example/dev/agents/claims-agent"
ok,_=authorize("claims-agent","claim.update","claim:483",bad,TASK)
tests.append(("wrong workload identity",ok,False))

bad=copy.deepcopy(posture)
bad["image_digest"]="sha256:evil"
ok,_=authorize("claims-agent","claim.update","claim:483",bad,TASK)
tests.append(("unapproved artifact",ok,False))

bad=copy.deepcopy(posture)
bad["workload_attested"]=False
ok,_=authorize("claims-agent","claim.update","claim:483",bad,TASK)
tests.append(("unattested workload",ok,False))

bad=copy.deepcopy(posture)
bad["status"]="quarantined"
ok,_=authorize("claims-agent","claim.update","claim:483",bad,TASK)
tests.append(("quarantined workload",ok,False))

for name,got,expected in tests:
    print(name,got)
    assert got==expected

print("all tests passed")


## 28 — Real SPIRE lab

Use the current official SPIRE deployment documentation.

Create a trust domain:

```text
corp.example
```

Register:

```text
spiffe://corp.example/prod/agents/claims-agent
```

with selectors appropriate to your runtime.

Then:

1. start SPIRE Server;
2. attest a SPIRE Agent/node;
3. register the claims workload;
4. connect the workload to the Workload API;
5. fetch an X.509-SVID;
6. inspect its SPIFFE ID and TTL;
7. rotate the SVID;
8. fetch a JWT-SVID for a specific audience;
9. validate the JWT-SVID from a separate service;
10. remove/change registration and observe issuance behavior.

The `spire/` directory contains a scaffold, not insecure magic configuration.


## 29 — X.509-SVID mTLS exercise

Build two services:

```text
claims-agent
claims-api
```

Assign:

```text
spiffe://corp.example/prod/agents/claims-agent
spiffe://corp.example/prod/apis/claims
```

Establish mTLS using Workload API-provided X.509-SVIDs.

The server must authorize the exact client SPIFFE ID after authenticating it.


## 30 — JWT-SVID exercise

Request a JWT-SVID for:

```text
audience = claims-api
```

Verify it at `claims-api`.

Then attempt to replay it at:

```text
payments-api
```

The second validation must fail due to audience mismatch.


## 31 — Sigstore/Cosign exercise

Build a container image for the training agent.

Then use Cosign to:

```text
sign
verify
attach/verify attestations
inspect signer identity
```

Create policy requiring:

```text
expected OIDC issuer
expected workflow/signer identity
approved repository
matching artifact digest
```

Do not accept an artifact merely because *someone* signed it.


## 32 — SLSA exercise

Create or obtain SLSA provenance for the training artifact.

Verify:

```text
subject digest
builder identity
source repository
build type
external parameters
attestation authenticity
```

Compare the requirements of:

```text
Build L1
Build L2
Build L3
```

Then inspect the SLSA 1.2 Source Track and identify which source controls would matter for an enterprise agent.


## 33 — OPA exercise

The course includes:

```text
policies/opa/workload.rego
```

Feed verified workload context into OPA.

Do not ask OPA to magically prove cryptographic evidence. Verify signatures/SVIDs/provenance at trusted boundaries, then pass trustworthy normalized facts to policy.


## 34 — Runtime drift exercise

Add:

```text
unexpected image
unexpected namespace
unexpected service account
stale posture
unapproved agent version
unapproved policy version
```

For each drift event decide:

```text
log only
constrain
step-up
quarantine
terminate
```

Tie the response to action risk.


## 35 — Review questions

1. What is the difference between logical agent identity and workload identity?
2. What is a SPIFFE ID?
3. What is a trust domain?
4. What is a trust bundle?
5. What is an SVID?
6. Compare X.509-SVID and JWT-SVID.
7. Why does SPIFFE guidance prefer X.509-SVID where practical?
8. Why is JWT-SVID audience validation critical?
9. What does the Workload API do?
10. Why doesn't a workload need a long-lived password to call the Workload API?
11. What is node attestation?
12. What is workload attestation?
13. What are selectors?
14. Why can weak selectors cause identity impersonation?
15. Why should leaf SVID fingerprints not become stable authorization identities?
16. How do short-lived SVIDs change revocation strategy?
17. Why is workload identity not authorization?
18. How do you bind a logical agent to an approved runtime?
19. Why are image digests preferable to mutable tags?
20. What is SLSA provenance?
21. What changed conceptually with SLSA 1.2's Build and Source tracks?
22. Why must provenance be verified against expectations?
23. What does Sigstore/Cosign contribute?
24. What is runtime drift?
25. Why is attestation freshness important?
26. What does SPIFFE Federation establish?
27. Why does federation not grant authorization?
28. Why can a cryptographically valid SVID still belong to a quarantined workload?
29. How should an agent registry relate logical identity, SPIFFE identity and artifact identity?
30. What evidence should a workload-aware authorization decision retain?

# Next course

## Intermediate 09 — Authorization Governance, Delegation & Least Privilege at Scale
